## 6. Xuat ket qua ra `result.csv` va `result_remove.csv`

In [1]:
# ── Imports ──────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

## 1. Load & Gộp bảng (Merge)

In [2]:
# Load raw data
transaction = pd.read_excel('../data/raw/transaction_info.xlsx')
user        = pd.read_excel('../data/raw/user_info.xlsx')
merchant    = pd.read_excel('../data/raw/merchant_info.xlsx')

print(f'transaction : {transaction.shape}')
print(f'user        : {user.shape}')
print(f'merchant    : {merchant.shape}')

# Gop bang
df = transaction.merge(user, on='userID', how='left')
df = df.merge(merchant, on='appID', how='left')

print(f'\nSau khi merge: {df.shape}')
df.head(3)

transaction : (301107, 14)
user        : (33686, 3)
merchant    : (748, 3)

Sau khi merge: (301107, 18)


,transID,userID,sof,platform,appID,deviceID,userIP,reqDate,amount,userChargeAmount,discountAmount,transStatus,campaignID,promotion_type,gender,created_account_date,report_cat,report_sub_cat
0,a3a41dc5672b84429dcea8d7b495ec3c,869ddd0dfb5335cba981750b64d86f5f,sof3,platform3,24,8738fc9f87128290ed8474dcb672231b,708f658693a2e43632e3953d7135502a,2022-10-23 11:36:40.380,48500,28500,20000,1,9945,voucher,male,2022-06-17,Goods_Transaction,Goods_Transaction_Platform
1,4771dea5fc29cbbff74750b8b827a68c,891db526845fb7c95ecfe6c8b187e380,sof3,platform1,748,NaN,891db526845fb7c95ecfe6c8b187e380,2022-10-22 18:13:05.372,49800,29800,20000,1,9945,voucher,male,2021-06-11,Goods_Transaction,Goods_Transaction_Platform
2,21072972ff937114bd96364f526cd9d0,b0b9e303379c38407ca75c526f1394d6,sof3,platform1,748,NaN,b0b9e303379c38407ca75c526f1394d6,2022-10-25 00:14:58.213,171175,151175,20000,1,9945,voucher,female,2021-06-04,Goods_Transaction,Goods_Transaction_Platform


## 2. Kiểm tra: discountAmount + userChargeAmount = amount?

In [3]:
# ── Kiểm tra công thức tài chính ──────────────────────────────────────────────
df['_amount_check'] = df['discountAmount'] + df['userChargeAmount'] - df['amount']

mismatch = df[df['_amount_check'] != 0]

print(f'Tổng số dòng       : {len(df):,}')
print(f'Dòng KHÔNG khớp    : {len(mismatch):,}')
print(f'Tỷ lệ khớp         : {(1 - len(mismatch)/len(df))*100:.4f}%')

if len(mismatch) > 0:
    print('\nCác dòng lệch (mẫu):')
    display(mismatch[['transID', 'amount', 'discountAmount', 'userChargeAmount', '_amount_check']].head(10))
else:
    print('✅ Tất cả dòng: discountAmount + userChargeAmount = amount')

# Ghi flag vào cột kiểm tra (True = khớp)
df['amount_valid'] = df['_amount_check'] == 0
df.drop(columns=['_amount_check'], inplace=True)

Tổng số dòng       : 301,107
Dòng KHÔNG khớp    : 2,299
Tỷ lệ khớp         : 99.2365%

Các dòng lệch (mẫu):


,transID,amount,discountAmount,userChargeAmount,_amount_check
71,977a103e97fee086da746ceba111f0e1,265540,10000,262851,7311
270,58890576ed38e45ad6fb2b009164783e,210349,10000,206556,6207
531,c4a8f92259eb8bd6e60e5106dc65aefd,1174551,10000,1190042,25491
610,b32140e80dccbbe46de55c594e1cd7cf,75000,20000,0,-55000
792,99bb201d4bf83e0b79c16da84e802642,40000,20000,0,-20000
883,a0ea0160face1bd8cdd73ad39ca5f48f,50000,20000,0,-30000
1023,6039edfa255be2c3661c14be94c18c71,45000,20000,0,-25000
1083,d11f735885858bb6844e39a9c91d0029,40000,20000,0,-20000
1085,c9a4ed8329db751fc1f863412ae513d3,40000,20000,0,-20000
1086,b7d48de17e22d2672367f11d3614e5ca,40000,20000,0,-20000


## 3. Điền `unknown` cho cột định danh (ID) bị null

In [4]:
# ── Cột định danh (ID / categorical string) ───────────────────────────────────
# Các cột được coi là "định danh" = kiểu object (string) hoặc tên chứa 'ID'
id_cols = [
    col for col in df.columns
    if df[col].dtype == object or col.lower().endswith('id')
]

print('Cột định danh được xử lý:', id_cols)
print('\nNull trước khi điền:')
print(df[id_cols].isnull().sum()[df[id_cols].isnull().sum() > 0])

# Điền 'unknown' cho null
df[id_cols] = df[id_cols].fillna('unknown')

print('\nNull sau khi điền:')
remaining = df[id_cols].isnull().sum()
print(remaining[remaining > 0] if remaining.sum() > 0 else '✅ Không còn null trong cột định danh')

Cột định danh được xử lý: ['transID', 'userID', 'sof', 'platform', 'appID', 'deviceID', 'userIP', 'reqDate', 'campaignID', 'promotion_type', 'gender', 'report_cat', 'report_sub_cat', 'amount_valid']

Null trước khi điền:
platform             767
deviceID           28547
promotion_type    187240
dtype: int64

Null sau khi điền:
✅ Không còn null trong cột định danh


## 4. Điền median theo `campaignID` cho cột số bị null

In [5]:
# ── Cột số cần điền median theo nhóm campaignID ───────────────────────────────
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

# Loại bỏ cột flag vừa tạo và cột khoá
exclude = ['transStatus', 'campaignID']
numeric_fill_cols = [c for c in numeric_cols if c not in exclude]

print('Cột số sẽ điền median theo campaignID:', numeric_fill_cols)
print('\nNull trước khi điền:')
null_before = df[numeric_fill_cols].isnull().sum()
print(null_before[null_before > 0] if null_before.sum() > 0 else 'Không có null trong cột số')

# Điền median theo từng nhóm campaignID
def fill_median_by_group(df, group_col, fill_cols):
    for col in fill_cols:
        if df[col].isnull().sum() > 0:
            group_median = df.groupby(group_col)[col].transform('median')
            # Nếu toàn nhóm đều null → dùng median tổng thể
            global_median = df[col].median()
            df[col] = df[col].fillna(group_median).fillna(global_median)
    return df

df = fill_median_by_group(df, 'campaignID', numeric_fill_cols)

print('\nNull sau khi điền:')
null_after = df[numeric_fill_cols].isnull().sum()
print(null_after[null_after > 0] if null_after.sum() > 0 else '✅ Không còn null trong cột số')

Cột số sẽ điền median theo campaignID: ['appID', 'amount', 'userChargeAmount', 'discountAmount']

Null trước khi điền:
Không có null trong cột số

Null sau khi điền:
✅ Không còn null trong cột số


## 5. Xử lý Duplicate theo `transID` (khoá chính)

In [6]:
# ── Kiểm tra & loại bỏ duplicate theo transID ────────────────────────────────
n_before = len(df)
n_dup    = df.duplicated(subset='transID', keep=False).sum()

print(f'Tổng số dòng trước: {n_before:,}')
print(f'Số dòng duplicate (transID): {n_dup:,}')

if n_dup > 0:
    print('\nMẫu các dòng duplicate:')
    display(df[df.duplicated(subset='transID', keep=False)]
            .sort_values('transID')
            .head(10))

# Giữ lại bản ghi đầu tiên của mỗi transID
df = df.drop_duplicates(subset='transID', keep='first')

n_after = len(df)
print(f'\nSố dòng sau khi loại duplicate: {n_after:,}')
print(f'Đã loại bỏ: {n_before - n_after:,} dòng')
print(f'\nNull tổng thể trong bộ dữ liệu cuối:')
remaining_nulls = df.isnull().sum()
print(remaining_nulls[remaining_nulls > 0] if remaining_nulls.sum() > 0 else '✅ Không còn null')

Tổng số dòng trước: 301,107
Số dòng duplicate (transID): 0

Số dòng sau khi loại duplicate: 301,107
Đã loại bỏ: 0 dòng

Null tổng thể trong bộ dữ liệu cuối:
✅ Không còn null


## 6. Xuất kết quả ra `result.csv`

In [7]:
# Tach 2 tap du lieu
# result       : giu lai tat ca du lieu
# result_clean : xoa cac dong amount lech

df_result = df.copy()
df_clean  = df[df['amount_valid']].copy()

print(f'Tong dong: {len(df):,}')
print(f'  result        : {len(df_result):,} dong (giu tat ca)')
print(f'  result_clean  : {len(df_clean):,} dong (da xoa amount lech)')

# Export
df_result.drop(columns=['amount_valid']).to_csv(
    '../data/processed/result.csv',
    index=False,
    encoding='utf-8-sig'
)

df_clean.drop(columns=['amount_valid']).to_csv(
    '../data/processed/result_clean.csv',
    index=False,
    encoding='utf-8-sig'
)

print('Da xuat:')
print(f'  ../data/processed/result.csv       -> {len(df_result):,} dong')
print(f'  ../data/processed/result_clean.csv -> {len(df_clean):,} dong')

display(df_clean.head(3))

Tong dong: 301,107
  result        : 301,107 dong (giu tat ca)
  result_clean  : 298,808 dong (da xoa amount lech)
Da xuat:
  ../data/processed/result.csv       -> 301,107 dong
  ../data/processed/result_clean.csv -> 298,808 dong


,transID,userID,sof,platform,appID,deviceID,userIP,reqDate,amount,userChargeAmount,discountAmount,transStatus,campaignID,promotion_type,gender,created_account_date,report_cat,report_sub_cat,amount_valid
0,a3a41dc5672b84429dcea8d7b495ec3c,869ddd0dfb5335cba981750b64d86f5f,sof3,platform3,24,8738fc9f87128290ed8474dcb672231b,708f658693a2e43632e3953d7135502a,2022-10-23 11:36:40.380,48500,28500,20000,1,9945,voucher,male,2022-06-17,Goods_Transaction,Goods_Transaction_Platform,True
1,4771dea5fc29cbbff74750b8b827a68c,891db526845fb7c95ecfe6c8b187e380,sof3,platform1,748,unknown,891db526845fb7c95ecfe6c8b187e380,2022-10-22 18:13:05.372,49800,29800,20000,1,9945,voucher,male,2021-06-11,Goods_Transaction,Goods_Transaction_Platform,True
2,21072972ff937114bd96364f526cd9d0,b0b9e303379c38407ca75c526f1394d6,sof3,platform1,748,unknown,b0b9e303379c38407ca75c526f1394d6,2022-10-25 00:14:58.213,171175,151175,20000,1,9945,voucher,female,2021-06-04,Goods_Transaction,Goods_Transaction_Platform,True


In [8]:
# =========================
# Xoa giao dich khong thanh cong
# transStatus != 1
# Ap dung cho ca 2 tap:
# - result      : giu amount lech, chi xoa fail
# - result_clean: vua amount hop le vua success
# =========================

# result: giu tat ca amount, chi xoa transaction fail
df_result = df[df['transStatus'] == 1].copy()

# result_clean:
# amount hop le + transaction success
df_clean = df[
    (df['amount_valid']) &
    (df['transStatus'] == 1)
].copy()

print(f'Tong dong ban dau : {len(df):,}')
print(f'result             : {len(df_result):,} dong (success only)')
print(f'result_clean       : {len(df_clean):,} dong (amount hop le + success)')

# Export
df_result.drop(columns=['amount_valid']).to_csv(
    '../data/processed/result.csv',
    index=False,
    encoding='utf-8-sig'
)

df_clean.drop(columns=['amount_valid']).to_csv(
    '../data/processed/result_clean.csv',
    index=False,
    encoding='utf-8-sig'
)

print('\nDa xuat:')
print(f'  ../data/processed/result.csv       -> {len(df_result):,} dong')
print(f'  ../data/processed/result_clean.csv -> {len(df_clean):,} dong')

display(df_clean.head(3))

Tong dong ban dau : 301,107
result             : 272,740 dong (success only)
result_clean       : 271,782 dong (amount hop le + success)

Da xuat:
  ../data/processed/result.csv       -> 272,740 dong
  ../data/processed/result_clean.csv -> 271,782 dong


,transID,userID,sof,platform,appID,deviceID,userIP,reqDate,amount,userChargeAmount,discountAmount,transStatus,campaignID,promotion_type,gender,created_account_date,report_cat,report_sub_cat,amount_valid
0,a3a41dc5672b84429dcea8d7b495ec3c,869ddd0dfb5335cba981750b64d86f5f,sof3,platform3,24,8738fc9f87128290ed8474dcb672231b,708f658693a2e43632e3953d7135502a,2022-10-23 11:36:40.380,48500,28500,20000,1,9945,voucher,male,2022-06-17,Goods_Transaction,Goods_Transaction_Platform,True
1,4771dea5fc29cbbff74750b8b827a68c,891db526845fb7c95ecfe6c8b187e380,sof3,platform1,748,unknown,891db526845fb7c95ecfe6c8b187e380,2022-10-22 18:13:05.372,49800,29800,20000,1,9945,voucher,male,2021-06-11,Goods_Transaction,Goods_Transaction_Platform,True
2,21072972ff937114bd96364f526cd9d0,b0b9e303379c38407ca75c526f1394d6,sof3,platform1,748,unknown,b0b9e303379c38407ca75c526f1394d6,2022-10-25 00:14:58.213,171175,151175,20000,1,9945,voucher,female,2021-06-04,Goods_Transaction,Goods_Transaction_Platform,True
